# STAT5293 Final Project Code

This notebook contains the final project workflow for data preparation, LoRA fine-tuning, evaluation, and result reproduction.

**Note:** Outputs have been cleared to make the notebook render correctly on GitHub. To reproduce the results, open this notebook in Google Colab with GPU runtime and run the cells in order.


# Step 1: Build and inspect the distillation dataset

In [ ]:
!pip -q install openai datasets tqdm pandas
!pip -q install transformers datasets peft trl accelerate bitsandbytes
!pip install -q datasets transformers peft trl bitsandbytes accelerate

In [ ]:
import os
import re
import json
import time
import torch
import gc
from tqdm import tqdm
import pandas as pd
from datasets import load_dataset, DatasetDict
from google.colab import userdata
from openai import OpenAI
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer
from transformers import pipeline
from peft import PeftModel
import matplotlib.pyplot as plt


In [ ]:
# Set API key
# get the open ai api
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")


In [ ]:
# Imports and configuration
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# Use any teacher model available in your account
TEACHER_MODEL = "gpt-4.1-mini"

# Start small for debugging first
N_TRAIN = 1000
N_VAL = 100
N_TEST = 100

OUTPUT_DIR = "/content/gsm8k_cot_data"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [ ]:
# Load GSM8K and sample a small subset
dataset = load_dataset("gsm8k", "main")

train_df = pd.DataFrame(dataset["train"])
test_df  = pd.DataFrame(dataset["test"])

# Shuffle reproducibly
train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)
test_df = test_df.sample(frac=1, random_state=42).reset_index(drop=True)

pilot_train = train_df.iloc[:N_TRAIN].copy()
pilot_val   = train_df.iloc[N_TRAIN:N_TRAIN+N_VAL].copy()
pilot_test  = test_df.iloc[:N_TEST].copy()

print("Train:", len(pilot_train))
print("Val:", len(pilot_val))
print("Test:", len(pilot_test))

In [ ]:
# Helper functions

def extract_gsm8k_final_answer(answer_text):
    """
    This function extracts the final answer from the original GSM8K dataset solution.

    In GSM8K, the "answer" field often contains:
    1. the full worked-out solution
    2. the final answer at the end, written like: #### 42

    Example:
    "Natalia sold 48 clips in April and 24 in May. #### 72"

    This function tries to pull out just the final number, such as "72".
    """

    # If the input is not a string, we cannot search through it
    if not isinstance(answer_text, str):
        return None

    # Look for the GSM8K pattern:
    # #### followed by optional spaces, then a number
    #
    # r"####\s*([-+]?\d[\d,\.]*)"
    # ####       -> exact characters ####
    # \s*        -> zero or more spaces
    # (...)      -> capture the number inside
    # [-+]?      -> optional + or - sign
    # \d         -> first digit
    # [\d,\.]*   -> remaining digits, commas, or decimal points
    match = re.search(r"####\s*([-+]?\d[\d,\.]*)", answer_text)

    # If that special GSM8K format is found, return the number
    if match:
        # group(1) is the captured number
        # replace(",", "") changes "1,234" into "1234"
        return match.group(1).replace(",", "")

    # If the #### pattern is missing, use a fallback:
    # find all numbers in the string and return the last one
    nums = re.findall(r"[-+]?\d[\d,\.]*", answer_text)
    if nums:
        return nums[-1].replace(",", "")

    # If no number is found at all, return None
    return None


def normalize_whitespace(text):
    """
    This function cleans up spacing in a piece of text.

    It replaces multiple spaces, tabs, or line breaks with a single space,
    and removes extra spaces at the beginning or end.

    Example:
    "Hello\\n\\n   world"  ->  "Hello world"
    """

    # If text is missing, return None
    if text is None:
        return None

    # \s+ means one or more whitespace characters
    # replace them with a single space, then strip outer spaces
    return re.sub(r"\s+", " ", text).strip()


# Ask the teacher model to produce: verbose reasoning & compressed reasoning
# This is Distillation targets for the student

def call_teacher(question, mode="verbose", max_retries=3):
    """
    This function sends one math question to the teacher model.

    Parameters:
    - question: the GSM8K math problem
    - mode:
        'verbose'    -> ask for full step-by-step reasoning
        'compressed' -> ask for short essential reasoning only
    - max_retries: how many times to retry if the API call fails

    Output:
    - the teacher model's response as cleaned text
    - or None if all retries fail
    """

    # Build the prompt differently depending on which reasoning style we want
    if mode == "verbose":
        prompt = f"""
Solve the following math word problem step by step.

Requirements:
- Show clear intermediate reasoning.
- Keep the reasoning readable and explicit.
- End with:
Answer: <final numeric answer only>

Question:
{question}
""".strip()

    elif mode == "compressed":
        prompt = f"""
Solve the following math word problem using concise reasoning.

Requirements:
- Include only the essential computational steps.
- Do not include extra explanation or redundant wording.
- End with:
Answer: <final numeric answer only>

Question:
{question}
""".strip()

    # If the mode is neither verbose nor compressed, stop with an error
    else:
        raise ValueError("mode must be 'verbose' or 'compressed'")

    # Try calling the model up to max_retries times
    for attempt in range(max_retries):
        try:
            response = client.responses.create(
                model=TEACHER_MODEL,
                input=prompt
            )

            # response.output_text is the plain text answer from the model
            text = response.output_text

            # Clean up extra spacing before returning
            return normalize_whitespace(text)

        except Exception as e:
            # If something goes wrong, print the error and wait a bit
            print(f"[Retry {attempt+1}/{max_retries}] Error: {e}")
            time.sleep(2 + attempt)

    # If all retries fail, return None
    return None


def extract_answer_from_model_output(text):
    """
    This function extracts the final answer from the teacher model's output.

    We ask the model to end with:
    Answer: 42

    So this function first tries to find that exact pattern.

    If it cannot find 'Answer: ...', it falls back to taking the last number
    that appears in the output.
    """

    # If text is empty or None, return None
    if not text:
        return None

    # First try to find:
    # Answer: 42
    #
    # re.IGNORECASE means it will also match "answer:" or "ANSWER:"
    match = re.search(r"Answer:\s*([-+]?\d[\d,\.]*)", text, re.IGNORECASE)
    if match:
        return match.group(1).replace(",", "")

    # Fallback: find all numbers and return the last one
    nums = re.findall(r"[-+]?\d[\d,\.]*", text)
    if nums:
        return nums[-1].replace(",", "")

    # If nothing usable is found, return None
    return None

In [ ]:
# Build one split
def build_split(df, split_name):
    records = []

    for idx, row in tqdm(df.iterrows(), total=len(df), desc=f"Building {split_name}"):
        question = row["question"]
        gold_full = row["answer"]
        gold_answer = extract_gsm8k_final_answer(gold_full)

        verbose_cot = call_teacher(question, mode="verbose")
        compressed_cot = call_teacher(question, mode="compressed")

        verbose_answer = extract_answer_from_model_output(verbose_cot)
        compressed_answer = extract_answer_from_model_output(compressed_cot)

        record = {
            "id": f"{split_name}_{idx}",
            "split": split_name,
            "question": question,
            "gold_solution": gold_full,
            "gold_answer": gold_answer,
            "verbose_cot": verbose_cot,
            "compressed_cot": compressed_cot,
            "verbose_answer": verbose_answer,
            "compressed_answer": compressed_answer
        }

        records.append(record)

    return records

In [ ]:
# Run generation
train_records = build_split(pilot_train, "train")
val_records   = build_split(pilot_val, "val")
test_records  = build_split(pilot_test, "test")

In [ ]:
# Save raw JSONL files
def save_jsonl(records, path):
    with open(path, "w", encoding="utf-8") as f:
        for r in records:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

save_jsonl(train_records, f"{OUTPUT_DIR}/train_raw.jsonl")
save_jsonl(val_records,   f"{OUTPUT_DIR}/val_raw.jsonl")
save_jsonl(test_records,  f"{OUTPUT_DIR}/test_raw.jsonl")

print("Saved raw files to:", OUTPUT_DIR)

In [ ]:
# Basic cleaning
def is_valid_record(r):
    return (
        r["question"] is not None and
        r["gold_answer"] is not None and
        r["verbose_cot"] is not None and
        r["compressed_cot"] is not None and
        r["verbose_answer"] is not None and
        r["compressed_answer"] is not None
    )

train_clean = [r for r in train_records if is_valid_record(r)]
val_clean   = [r for r in val_records if is_valid_record(r)]
test_clean  = [r for r in test_records if is_valid_record(r)]

print("Raw train:", len(train_records), "Clean train:", len(train_clean))
print("Raw val:", len(val_records), "Clean val:", len(val_clean))
print("Raw test:", len(test_records), "Clean test:", len(test_clean))

In [ ]:
# Save final clean JSONL
save_jsonl(train_clean, f"{OUTPUT_DIR}/train_clean.jsonl")
save_jsonl(val_clean,   f"{OUTPUT_DIR}/val_clean.jsonl")
save_jsonl(test_clean,  f"{OUTPUT_DIR}/test_clean.jsonl")

print("Saved clean files.")

In [ ]:
# Build the 3 training formats
# 1. answer-only
# 2. verbose CoT
# 3. compressed CoT

def format_answer_only(r):
    return {
        "id": r["id"],
        "split": r["split"],
        "input": f"Question: {r['question']}",
        "output": f"Answer: {r['gold_answer']}"
    }

def format_verbose(r):
    return {
        "id": r["id"],
        "split": r["split"],
        "input": f"Question: {r['question']}",
        "output": f"{r['verbose_cot']}"
    }

def format_compressed(r):
    return {
        "id": r["id"],
        "split": r["split"],
        "input": f"Question: {r['question']}",
        "output": f"{r['compressed_cot']}"
    }

train_answer_only = [format_answer_only(r) for r in train_clean]
train_verbose     = [format_verbose(r) for r in train_clean]
train_compressed  = [format_compressed(r) for r in train_clean]

val_answer_only = [format_answer_only(r) for r in val_clean]
val_verbose     = [format_verbose(r) for r in val_clean]
val_compressed  = [format_compressed(r) for r in val_clean]

test_answer_only = [format_answer_only(r) for r in test_clean]
test_verbose     = [format_verbose(r) for r in test_clean]
test_compressed  = [format_compressed(r) for r in test_clean]

In [ ]:
save_jsonl(train_answer_only, f"{OUTPUT_DIR}/train_answer_only.jsonl")
save_jsonl(train_verbose,     f"{OUTPUT_DIR}/train_verbose.jsonl")
save_jsonl(train_compressed,  f"{OUTPUT_DIR}/train_compressed.jsonl")

save_jsonl(val_answer_only, f"{OUTPUT_DIR}/val_answer_only.jsonl")
save_jsonl(val_verbose,     f"{OUTPUT_DIR}/val_verbose.jsonl")
save_jsonl(val_compressed,  f"{OUTPUT_DIR}/val_compressed.jsonl")

save_jsonl(test_answer_only, f"{OUTPUT_DIR}/test_answer_only.jsonl")
save_jsonl(test_verbose,     f"{OUTPUT_DIR}/test_verbose.jsonl")
save_jsonl(test_compressed,  f"{OUTPUT_DIR}/test_compressed.jsonl")

print("Saved all dataset variants.")

In [ ]:
# Inspect a few examples
for i in range(3):
    r = train_clean[i]
    print("=" * 100)
    print("QUESTION:")
    print(r["question"])
    print("\nGOLD ANSWER:", r["gold_answer"])
    print("\nVERBOSE COT:")
    print(r["verbose_cot"])
    print("\nCOMPRESSED COT:")
    print(r["compressed_cot"])

In [ ]:
# Quick summary stats
def token_proxy_len(text):
    if text is None:
        return 0
    return len(text.split())

summary = {
    "train_examples_clean": len(train_clean),
    "avg_verbose_len": sum(token_proxy_len(r["verbose_cot"]) for r in train_clean) / max(len(train_clean), 1),
    "avg_compressed_len": sum(token_proxy_len(r["compressed_cot"]) for r in train_clean) / max(len(train_clean), 1),
    "verbose_matches_gold": sum(r["verbose_answer"] == r["gold_answer"] for r in train_clean),
    "compressed_matches_gold": sum(r["compressed_answer"] == r["gold_answer"] for r in train_clean),
}

summary

In [ ]:
# Find missed question for verbose cot and compressed cot
for r in train_clean:
    verbose_wrong = (r["verbose_answer"] != r["gold_answer"])
    compressed_wrong = (r["compressed_answer"] != r["gold_answer"])

    if verbose_wrong or compressed_wrong:
        print("=" * 100)
        print("ID:", r["id"])
        print("QUESTION:")
        print(r["question"])
        print("\nGOLD ANSWER:", r["gold_answer"])
        print("\nVERBOSE ANSWER:", r["verbose_answer"], "| Wrong?", verbose_wrong)
        print("COMPRESSED ANSWER:", r["compressed_answer"], "| Wrong?", compressed_wrong)
        print("\nVERBOSE OUTPUT:")
        print(r["verbose_cot"])
        print("\nCOMPRESSED OUTPUT:")
        print(r["compressed_cot"])

# Step 2: Fine-tune student models

In [ ]:
# choose one dataset version to train
DATA_DIR = "/content/gsm8k_cot_data"

TRAIN_FILE = f"{DATA_DIR}/train_compressed.jsonl"
VAL_FILE   = f"{DATA_DIR}/val_compressed.jsonl"

# Later you can swap these for:
# train_answer_only.jsonl / val_answer_only.jsonl
# train_verbose.jsonl / val_verbose.jsonl

In [ ]:
# load the dataset
dataset = load_dataset(
    "json",
    data_files={
        "train": TRAIN_FILE,
        "validation": VAL_FILE,
    }
)

dataset

In [ ]:
def to_prompt_completion(example):
    return {
        "prompt": example["input"] + "\n",
        "completion": example["output"]
    }

dataset = dataset.map(to_prompt_completion)
dataset

In [ ]:
# Select small student model
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"

In [ ]:
# tokenizer + 4-bit model loading
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)

# TinyLlama usually has an eos token; set pad token if missing
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

model.config.use_cache = False

In [ ]:
# LoRA config
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

In [ ]:
# Train all 3 model together
conditions = ["answer_only", "verbose", "compressed"]
all_train_loss_dfs = []
all_val_loss_dfs = []

for condition in conditions:
    print("=" * 80)
    print(f"Training condition: {condition}")

    TRAIN_FILE = f"{DATA_DIR}/train_{condition}.jsonl"
    VAL_FILE   = f"{DATA_DIR}/val_{condition}.jsonl"
    output_dir = f"/content/student_{condition}"

    # Load dataset for this condition
    dataset = load_dataset(
        "json",
        data_files={
            "train": TRAIN_FILE,
            "validation": VAL_FILE,
        }
    )
    dataset = dataset.map(to_prompt_completion)

    # Load tokenizer
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, use_fast=True)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    # Load a fresh base model for this condition
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16,
    )
    model.config.use_cache = False

    # Training config MUST be inside the loop
    training_args = SFTConfig(
        output_dir=output_dir,
        per_device_train_batch_size=1,
        per_device_eval_batch_size=1,
        gradient_accumulation_steps=2,
        learning_rate=2e-4,
        num_train_epochs=1,
        max_steps=100,
        logging_steps=1,
        eval_strategy="steps",
        eval_steps=5,
        save_strategy="no",
        fp16=False,
        bf16=False,
        max_length=256,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        weight_decay=0.01,
        report_to="none",
        completion_only_loss=True,
        dataset_num_proc=1,
    )

    trainer = SFTTrainer(
        model=model,
        args=training_args,
        train_dataset=dataset["train"],
        eval_dataset=dataset["validation"],
        processing_class=tokenizer,
        peft_config=peft_config,
    )

    trainer.can_return_loss = True
    trainer.train()

    log_history = trainer.state.log_history

    train_logs = [x for x in log_history if "loss" in x and "step" in x]
    eval_logs = [x for x in log_history if "eval_loss" in x and "step" in x]

    train_loss_df = pd.DataFrame({
        "Step": [x["step"] for x in train_logs],
        "Training Loss": [x["loss"] for x in train_logs],
        "condition": condition
    })

    val_loss_df = pd.DataFrame({
        "Step": [x["step"] for x in eval_logs],
        "Validation Loss": [x["eval_loss"] for x in eval_logs],
        "condition": condition
    })

    train_loss_df.to_csv(f"{output_dir}/train_loss_history.csv", index=False)
    val_loss_df.to_csv(f"{output_dir}/val_loss_history.csv", index=False)

    all_train_loss_dfs.append(train_loss_df)
    all_val_loss_dfs.append(val_loss_df)

    trainer.model.save_pretrained(output_dir)
    tokenizer.save_pretrained(output_dir)

    print(f"Saved model to {output_dir}")

    # Clear memory before next condition
    del trainer, model, tokenizer, dataset
    gc.collect()
    torch.cuda.empty_cache()
all_train_loss = pd.concat(all_train_loss_dfs, ignore_index=True)
all_val_loss = pd.concat(all_val_loss_dfs, ignore_index=True)

all_train_loss.head()
all_val_loss.head()

In [ ]:
conditions = ["answer_only", "verbose", "compressed"]

train_dfs = []
val_dfs = []

for condition in conditions:
    output_dir = f"/content/student_{condition}"
    train_path = f"{output_dir}/train_loss_history.csv"
    val_path = f"{output_dir}/val_loss_history.csv"

    print(condition)
    print(" train exists:", os.path.exists(train_path))
    print(" val exists:", os.path.exists(val_path))

    if os.path.exists(train_path):
        train_dfs.append(pd.read_csv(train_path))
    if os.path.exists(val_path):
        val_dfs.append(pd.read_csv(val_path))

all_train_loss = pd.concat(train_dfs, ignore_index=True)
all_val_loss = pd.concat(val_dfs, ignore_index=True)

display(all_train_loss.head())
display(all_val_loss.head())

In [ ]:
# plot compressed first

cond = "compressed"

train_df = all_train_loss[all_train_loss["condition"] == cond]
val_df = all_val_loss[all_val_loss["condition"] == cond]

plt.figure(figsize=(8,5))
plt.plot(train_df["Step"], train_df["Training Loss"], marker="o", label="Training Loss")
plt.plot(val_df["Step"], val_df["Validation Loss"], marker="s", label="Validation Loss")
plt.xlabel("Training Step")
plt.ylabel("Loss")
plt.title(f"Training and Validation Loss ({cond})")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))

for condition in all_train_loss["condition"].unique():
    sub = all_train_loss[all_train_loss["condition"] == condition]
    plt.plot(sub["Step"], sub["Training Loss"], label=condition)

plt.xlabel("Step")
plt.ylabel("Training Loss")
plt.title("Training Loss Across Distillation Conditions")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
all_train_loss["Moving_Avg_3"] = (
    all_train_loss
    .groupby("condition")["Training Loss"]
    .transform(lambda x: x.rolling(window=3).mean())
)

plt.figure(figsize=(8, 5))

for condition in all_train_loss["condition"].unique():
    sub = all_train_loss[all_train_loss["condition"] == condition]

    plt.plot(
        sub["Step"],
        sub["Training Loss"],
        marker="o",
        alpha=0.35,
        label=f"{condition} raw"
    )

    plt.plot(
        sub["Step"],
        sub["Moving_Avg_3"],
        linewidth=2,
        label=f"{condition} 3-step MA"
    )

plt.xlabel("Training Step")
plt.ylabel("Training Loss")
plt.title("Training Loss During Fine-Tuning")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

Over 50 training steps, the loss exhibited short-term fluctuations but an overall downward trajectory. Starting at 1.1937 in the first step, the loss gradually declined and reached 0.2630 by step 50. Several later steps achieved particularly low loss values, such as 0.1103 at step 43 and 0.0841 at step 48, indicating substantial improvement over the course of training. Despite some instability across individual steps, the general trend suggests that the fine-tuning process was successful and that the student model was learning from the teacher-generated reasoning data.

In [ ]:
# 1) answer-only baseline
TRAIN_FILE = f"{DATA_DIR}/train_answer_only.jsonl"
VAL_FILE   = f"{DATA_DIR}/val_answer_only.jsonl"
output_dir = "/content/student_answer_only"

# 2) verbose CoT
TRAIN_FILE = f"{DATA_DIR}/train_verbose.jsonl"
VAL_FILE   = f"{DATA_DIR}/val_verbose.jsonl"
output_dir = "/content/student_verbose"

# 3) compressed CoT
TRAIN_FILE = f"{DATA_DIR}/train_compressed.jsonl"
VAL_FILE   = f"{DATA_DIR}/val_compressed.jsonl"
output_dir = "/content/student_compressed"

# Step 3: Evaluate and Compare Student Models

for each question in test set:

    1. generate model output
    2. extract predicted answer
    3. compare with gold answer
    record:
        - correctness
        - output length
        - latency

In [ ]:
print("Exists?", os.path.exists("/content/student_answer_only"))
print("Exists?", os.path.exists("/content/student_compressed"))
print("Exists?", os.path.exists("/content/student_verbose"))

In [ ]:
MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
adapter_path = "/content/student_compressed"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(adapter_path)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.float16,
)

model = PeftModel.from_pretrained(base_model, adapter_path)
model.eval()

In [ ]:
# Generate the answer
def generate_answer(model, tokenizer, question, max_new_tokens=256):
    # Match the format used during training
    prompt = f"Question: {question}\n"

    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    input_len = inputs["input_ids"].shape[1]

    start = time.time()
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    latency = time.time() - start

    # Decode only the newly generated part
    new_tokens = outputs[0][input_len:]
    generated_text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    return generated_text, latency

In [ ]:
# Define answer extractor


def extract_answer_from_model_output(text):
    if not text or not text.strip():
        return None

    match = re.search(r"Answer:\s*([-+]?\d[\d,\.]*)", text, re.IGNORECASE)
    if match:
        return match.group(1).replace(",", "")

    match = re.search(r"answer\s+is\s+([-+]?\d[\d,\.]*)", text, re.IGNORECASE)
    if match:
        return match.group(1).replace(",", "")

    nums = re.findall(r"[-+]?\d[\d,\.]*", text)
    if nums:
        return nums[-1].replace(",", "")

    return None

In [ ]:
# Load GSM8K test set
test_raw = load_dataset("gsm8k", "main", split="test")

def extract_gold_answer(answer_text):
    # GSM8K answer format usually has final answer after ####
    match = re.search(r"####\s*([-+]?\d*\.?\d+)", answer_text.replace(",", ""))
    if match:
        return match.group(1)
    return None

# Make a clean test list
test_clean = []

for ex in test_raw:
    gold = extract_gold_answer(ex["answer"])
    if gold is not None:
        test_clean.append({
            "question": ex["question"],
            "gold_answer": gold,
            "full_answer": ex["answer"]
        })

print("Number of test examples:", len(test_clean))
print(test_clean[0])

In [ ]:
N_TEST = 20
conditions = ["answer_only", "verbose", "compressed"]

all_eval_rows = []

for condition in conditions:
    print("=" * 80)
    print(f"Evaluating condition: {condition}")

    adapter_path = f"/content/student_{condition}"

    tokenizer = AutoTokenizer.from_pretrained(adapter_path)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16,
    )

    model = PeftModel.from_pretrained(base_model, adapter_path)
    model.eval()

    for i, r in enumerate(test_clean[:N_TEST]):
        print(f"{condition}: {i+1}/{N_TEST}")

        generated_text, latency = generate_answer(model, tokenizer, r["question"])
        pred_answer = extract_answer_from_model_output(generated_text)

        all_eval_rows.append({
            "condition": condition,
            "question": r["question"],
            "gold_answer": r["gold_answer"],
            "pred_answer": pred_answer,
            "correct": int(pred_answer == r["gold_answer"]),
            "output_length": len(generated_text.split()),
            "latency_sec": latency,
            "generated_text": generated_text
        })

    del model, base_model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

all_eval_df = pd.DataFrame(all_eval_rows)

summary_df = all_eval_df.groupby("condition").agg(
    accuracy=("correct", "mean"),
    avg_latency_sec=("latency_sec", "mean"),
    avg_output_length=("output_length", "mean")
).reset_index()

summary_df

In [ ]:
N_TEST = 50
conditions = ["answer_only", "verbose", "compressed"]

all_eval_rows = []

for condition in conditions:
    print("=" * 80)
    print(f"Evaluating condition: {condition}")

    adapter_path = f"/content/student_{condition}"

    tokenizer = AutoTokenizer.from_pretrained(adapter_path)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    base_model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16,
    )

    model = PeftModel.from_pretrained(base_model, adapter_path)
    model.eval()

    for i, r in enumerate(test_clean[:N_TEST]):
        print(f"{condition}: {i+1}/{N_TEST}")

        generated_text, latency = generate_answer(model, tokenizer, r["question"])
        pred_answer = extract_answer_from_model_output(generated_text)

        all_eval_rows.append({
            "condition": condition,
            "question": r["question"],
            "gold_answer": r["gold_answer"],
            "pred_answer": pred_answer,
            "correct": int(pred_answer == r["gold_answer"]),
            "output_length": len(generated_text.split()),
            "latency_sec": latency,
            "generated_text": generated_text
        })

    del model, base_model, tokenizer
    gc.collect()
    torch.cuda.empty_cache()

all_eval_df = pd.DataFrame(all_eval_rows)

summary_df = all_eval_df.groupby("condition").agg(
    accuracy=("correct", "mean"),
    avg_latency_sec=("latency_sec", "mean"),
    avg_output_length=("output_length", "mean")
).reset_index()

summary_df

In [ ]:
all_eval_df["condition"].value_counts()

Compared with verbose CoT, compressed CoT reduced average latency from 29.27s to 10.80s and reduced average output length from 127.62 tokens to 41.74 tokens, while only decreasing accuracy from 68% to 64%. This suggests that compressed reasoning supervision preserves much of the reasoning benefit of verbose CoT while improving inference efficiency.